In [2]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

import os
current_pwd = os.getcwd()
possible_paths = [
    '/home/export/soheuny/SRFinder/soheun/notebooks', 
    '/home/soheuny/HH4bsim/soheun/notebooks'
]
    
assert os.getcwd() in possible_paths, f"Did you change the path? It should be one of {possible_paths}"
os.chdir("..")

In [3]:
from fvt_classifier import FvTClassifier
from copy import deepcopy
from training_info import TrainingInfo
from dataset import MotherSamples
import numpy as np
from constants import FEATURES
import torch
import pytorch_lightning as pl

config = {
    "experiment_name": "base_fvt_training_ZH4b",
    "dataset": {
        "signal_filename": "ZH4b_picoAOD.h5",
        "signal_ratio": 0.02,
        "n_3b": 100_0000,
        "ratio_4b": 0.5,
        "seed": 0,
        "base_fvt_train_ratio": 0.5,
    },
    "base_fvt": {
        "model": FvTClassifier,
        "dim_dijet_features": 6,
        "dim_quadjet_features": 6,
        "repr_norm": False,
        "depth": {
            "encoder": 4,
            "decoder": 1,
        },
        "fit_batch_size": 1024,
        "model_seed": 0,
        "train_seed": 0,
        "data_seed": 0,
        "max_epochs": 100,
        "val_ratio": 0.33,
        "early_stop_patience": None,
        "optimizer": {
            "type": "Adam",
            "lr": 0.01,
        },
        "lr_scheduler": {
            "type": "ReduceLROnPlateau",
            "factor": 0.5,
            "threshold": 0.0001,
            "patience": 10,
            "cooldown": 1,
            "min_lr": 0.0002,
        },
        "dataloader": {
            "batch_size": 1024,
            "batch_size_multiplier": 2,
            "batch_size_milestones": [1, 3, 6, 10, 15],
        },
    }
}

    

signal_ratio = config["dataset"]["signal_ratio"]
n_3b = config["dataset"]["n_3b"]
ratio_4b = config["dataset"]["ratio_4b"]
signal_filename = config["dataset"]["signal_filename"]
seed = config["dataset"]["seed"]
base_fvt_train_ratio = config["dataset"]["base_fvt_train_ratio"]

base_fvt_hparams = deepcopy(config["base_fvt"])
base_fvt_hparams["experiment_name"] = config["experiment_name"]
base_fvt_hparams["dataset"] = config["dataset"]
base_fvt_hparams["step"] = 1

# Unchangeable configurations
dim_input_jet_features = 4
num_classes = 2

# 1. Find and load the mother dataset
ms_hparams = {
    "n_3b": n_3b,
    "ratio_4b": ratio_4b,
    "signal_ratio": signal_ratio,
    "signal_filename": signal_filename,
    "seed": seed,
}
hashes = MotherSamples.find(ms_hparams, from_metadata=False)
if len(hashes) == 0:
    raise ValueError(
        "No mother samples found for the given parameters, first save the mother samples with hparams {}".format(
            ms_hparams
        )
    )
elif len(hashes) > 1:
    raise ValueError(
        "Number of mother samples must be one, instead of {}".format(len(hashes))
    )
ms_hash = hashes[0]
mother_samples = MotherSamples.load(ms_hash)

# 2. Split the mother dataset into train and test
# Train -- validation split will be done by TrainingInfoV2

ms_len = len(mother_samples.scdinfo)
ms_idx = np.zeros(ms_len, dtype=bool)
ms_idx[: int(ms_len * base_fvt_train_ratio)] = True
np.random.seed(seed)
np.random.shuffle(ms_idx)

base_fvt_tinfo = TrainingInfo(base_fvt_hparams, ms_hash=ms_hash, ms_idx=ms_idx)
print("Base FvT Training Hash: ", base_fvt_tinfo.hash)

base_fvt_train_dset, base_fvt_val_dset = (
    base_fvt_tinfo.fetch_train_val_tensor_datasets(FEATURES, "fourTag", "weight")
)

model_seed = base_fvt_hparams["model_seed"]
pl.seed_everything(model_seed)
base_fvt_model = FvTClassifier(
    num_classes,
    dim_input_jet_features,
    base_fvt_hparams["dim_dijet_features"],
    base_fvt_hparams["dim_quadjet_features"],
    run_name=base_fvt_tinfo.hash,
    device=torch.device("cuda:0"),
    depth=base_fvt_hparams["depth"],
    repr_norm=base_fvt_hparams["repr_norm"],
)

1850it [00:03, 464.64it/s]


Base FvT Training Hash:  250823_203930_166152_NEvqej


[rank: 0] Seed set to 0


In [22]:
base_fvt_model.forward(base_fvt_val_dset[128:160][0].to(torch.device("cuda:0")))

NaN found in forward: q
x tensor([[ 1.3166e+02,  1.2051e+02,  1.1819e+02,  4.9161e+01,  6.0252e-01,
          1.3854e+00, -1.3201e+00,  1.6926e+00,  0.0000e+00,  2.5367e+00,
          3.0451e+00, -2.6889e+00,  0.0000e+00,  0.0000e+00,  0.0000e+00,
          0.0000e+00],
        [ 1.1582e+02,  7.9654e+01,  7.2730e+01,  7.0674e+01,  8.6190e-01,
          6.8941e-01, -1.3516e+00,  1.4041e-01,  0.0000e+00,  3.0824e+00,
          1.2733e+00, -2.2167e+00,  0.0000e+00,  0.0000e+00,  0.0000e+00,
          0.0000e+00],
        [ 1.6380e+02,  8.1410e+01,  4.3152e+01,  4.0880e+01,  2.5275e-01,
         -1.4880e+00,  1.0267e+00, -6.0552e-01,  0.0000e+00,  2.6762e+00,
         -4.9151e-01,  3.0391e+00,  0.0000e+00,  0.0000e+00,  0.0000e+00,
          0.0000e+00],
        [ 1.0024e+02,  9.6329e+01,  6.2950e+01,  5.6368e+01,  8.7657e-01,
          1.0970e+00,  2.3162e+00, -2.0099e+00,  0.0000e+00,  2.0260e+00,
         -2.9073e+00, -1.5379e+00,  0.0000e+00,  0.0000e+00,  0.0000e+00,
          0.0000e

ValueError: NaN found in forward: q

In [14]:
base_fvt_val_dset[155][0]

tensor([91.3389, 90.6404, 90.6404, 50.0490,  0.5292,  1.1995,  1.1995,  2.1270,
         0.0000,  3.0841,  3.0841, -1.6995,  0.0000,  0.0000,  0.0000,  0.0000])

In [13]:
from ancillary_features import get_ancillary_features

get_ancillary_features(base_fvt_val_dset[155][0])

(tensor([[91.3389, 90.6404, 90.6404, 50.0490, 91.3389, 90.6404, 90.6404, 50.0490,
          91.3389, 50.0490, 90.6404, 90.6404,  0.5292,  1.1995,  1.1995,  2.1270],
         [ 0.5292,  1.1995,  1.1995,  2.1270,  0.5292,  2.1270,  1.1995,  1.1995,
           0.0000,  3.0841,  3.0841, -1.6995,  0.0000,  3.0841,  3.0841, -1.6995],
         [ 0.0000, -1.6995,  3.0841,  3.0841,  0.0000,  0.0000,  0.0000,  0.0000,
           0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000]]),
 tensor([[[192.2229],
          [112.3299],
          [192.2229],
          [112.3299],
          [157.1314],
          [  0.0000],
          [  3.1561],
          [  1.7633],
          [  3.1561],
          [  1.7633],
          [  2.3326],
          [  0.0000]]]),
 tensor([[[  3.2126],
          [  3.2126],
          [  2.7149],
          [351.8886],
          [351.8886],
          [351.8886]]]))